# Giai đoạn 10: Đánh giá Đỉnh cao (Tập Dữ Liệu Tối Ưu 13 Đặc Trưng)
Áp dụng nguyên lý "Less is More": Loại bỏ 3 đặc trưng tần số phái sinh gây nhiễu (`LF_HF_Ratio`, `LF_norm`, `HF_norm`), chỉ giữ lại **13 đặc trưng gốc rễ** mạnh nhất.
Chúng ta sẽ cho 12 mô hình (bao gồm TabPFN) so tài lại trên bộ dữ liệu hoàn hảo này để xem LightGBM có trở lại ngôi vương 97.51% không, và các thuật toán khác sẽ thể hiện ra sao.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

try:
    from tabpfn import TabPFNClassifier
except ImportError:
    print("Đang cài đặt TabPFN...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tabpfn'])
    from tabpfn import TabPFNClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
import xgboost as xgb
import lightgbm as lgb

### 1. Đọc dữ liệu và LOẠI BỎ các đặc trưng gây nhiễu (Chỉ giữ lại 13 Đặc trưng Vàng)

In [ ]:
data_path = '../../data/features/mimic_train_features_advanced.csv'
df = pd.read_csv(data_path)

# ÁP DỤNG CHIẾN LƯỢC TỐI ƯU: Bỏ 3 đặc trưng phái sinh
cols_to_drop = ['LF_HF_Ratio', 'LF_norm', 'HF_norm']
df_optimal = df.drop(columns=cols_to_drop)

print(f"Đã loại bỏ: {cols_to_drop}")
print(f"Số lượng đặc trưng hiện tại: {df_optimal.shape[1] - 1} (Đã trừ cột status)")

# Lưu lại tập dữ liệu tối ưu này để dùng vĩnh viễn về sau
df_optimal.to_csv('../../data/features/mimic_train_features_optimal_13.csv', index=False)
print("Đã lưu tập dữ liệu tối ưu vào: mimic_train_features_optimal_13.csv\n")

X = df_optimal.drop(columns=['status'])
y = df_optimal['status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Tập Train: {len(X_train)} mẫu | Tập Test: {len(X_test)} mẫu")

### 2. Định nghĩa 12 mô hình (Đã fix lỗi tham số của TabPFN)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    "Extra Trees": ExtraTreesClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    "SVM": SVC(random_state=42, probability=True),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),
    "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    "LightGBM": lgb.LGBMClassifier(random_state=42, verbose=-1, n_estimators=300, learning_rate=0.05, num_leaves=31),
    "TabPFN": TabPFNClassifier(device='cpu') # Đã xoá N_ensemble_configurations để tránh lỗi phiên bản mới
}

results = []
confusion_matrices = {}

os.makedirs('../../models/optimal_compare', exist_ok=True)

for name, model in models.items():
    print(f"Đang huấn luyện: {name}...")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    
    # Có thể TabPFN sẽ đòi License, nếu có lỗi, ta sẽ bắt lỗi và skip nó
    try:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1
        })
        
        confusion_matrices[name] = confusion_matrix(y_test, y_pred)
        joblib.dump(pipeline, f"../../models/optimal_compare/{name.replace(' ', '_').lower()}.pkl")
    except Exception as e:
        print(f"❌ Mô hình {name} bị lỗi: {e}")
        print("Đã bỏ qua mô hình này.")

print("\n✅ Hoàn tất vòng huấn luyện!")

### 3. Vẽ Lưới Confusion Matrix

In [ ]:
import math
cmaps = ['Blues', 'Greens', 'Oranges', 'Purples', 'Reds', 
         'YlGnBu', 'YlOrBr', 'PuBu', 'GnBu', 'BuPu', 'Greys', 'PuRd']

num_models = len(confusion_matrices)
rows = math.ceil(num_models / 4)
cols = min(4, num_models)

fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4.5))
if num_models > 1: axes = axes.flatten()
else: axes = [axes]

for i, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmaps[i % len(cmaps)], ax=axes[i], cbar=False)
    axes[i].set_title(name, fontweight='bold', fontsize=13)
    axes[i].set_ylabel('Thực tế (True)', fontsize=10)
    axes[i].set_xlabel('Dự đoán (Predicted)', fontsize=10)
    axes[i].set_xticklabels(['Bình thường', 'AFib'])
    axes[i].set_yticklabels(['Bình thường', 'AFib'])

# Xoá ô thừa
for j in range(i + 1, rows * cols):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### 4. Bảng Xếp Hạng Accuracy (13 Đặc Trưng Tối Ưu)

In [ ]:
from IPython.display import display 
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

display(results_df)

plt.figure(figsize=(12, 7))
sns.barplot(data=results_df, x="Accuracy", y="Model", palette="plasma")
plt.title("Bảng xếp hạng trên Tập Dữ Liệu TỐI ƯU NHẤT (13 Đặc Trưng)", fontsize=15, fontweight='bold')
plt.xlabel("Accuracy (Độ chính xác)")
plt.ylabel("")
plt.xlim([0.8, 1.0])
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()